# The Program Desk — guided walkthrough

This notebook is your **learning trail**: it imports the real project modules
(`config.py`, `vector_store.py`, `tools.py`, `agent.py`) and runs the pipeline
one stage at a time so you can *see* what each piece does.

The app itself runs with `streamlit run app.py` — this notebook is for understanding it.

**The pipeline (offline part):** PDF → parse → chunk → embed → Chroma  
**The loop (live part):** question → agent → `search_gao_reports` → chunks → cited answer

> Run cells top to bottom. Prerequisite: `python ingest_gao.py` has been run once.

## 1. Configuration — one file rules them all

Every setting lives in `config.py`. Change the model tier or chunk size there and every module follows.

In [14]:
import config

print("Model:          ", config.MODEL_NAME)
print("Chunk size:     ", config.PAGES_PER_CHUNK, "pages")
print("Retrieval top-k:", config.TOP_K)
print("PDF:            ", config.GAO_PDF_PATH.name, "| exists:", config.GAO_PDF_PATH.exists())

Model:           gpt-4o-mini
Chunk size:      2 pages
Retrieval top-k: 4
PDF:             gao-26-108457.pdf | exists: True


In [15]:
import sys; print(sys.executable)

c:\Users\sol_v\OneDrive\Escritorio\IRON_AI\langchain-env\Scripts\python.exe


In [16]:
#%pip install flashrank

In [17]:
import langchain
print(langchain.__version__)


1.3.16


In [18]:
#%pip install pypdf

## 2. Parsing — what the raw PDF text looks like

`pypdf` extracts plain text page by page. Notice it's messy — headers, columns
flattened into lines. Embeddings tolerate this surprisingly well.

In [19]:
from pypdf import PdfReader

reader = PdfReader(str(config.GAO_PDF_PATH))
print(f"The report has {len(reader.pages)} pages.\n")

# Peek at a page from the middle of the report (program-profile territory)
sample_page = 120
print(reader.pages[sample_page].extract_text()[:800])

The report has 249 pages.

Army Program Type: MDAP Common Name: ITEP 
 
Page 109 U.S. Government Accountability Office GAO-26-108457  Weapon Systems Annual Assessment 
 
Improved Turbine Engine Program (ITEP) 
The Army’s ITEP is developing a next generation turbo-shaft engine for 
the Black Hawk and Apache helicopter fleets. The program includes 
engine development, manufacturing, platform integration, and 
qualification. The Army intends for the engine to fit inside the existing 
engine compartments of the Black Hawk and Apache helicopters and 
expects that the engine will provide power, fuel efficiency, reliability, 
and sustainment improvements. 
Source: U.S. Army.  |  GAO-26-108457  
 
 
 
Program Performance fiscal year 2025 dollars in millions 
 
 
aGAO-25-107569. 
 
Software Development 
as of January 2026 
A


## 3. Chunking — one chunk ≈ one program profile

A *chunk* is the unit of retrieval: small enough to be specific, big enough to
carry context. GAO profiles are standardized ~2-page spreads, so we cut every
2 pages and keep **metadata** (section name + page numbers) — that metadata is
exactly what the agent's citations are built from later.

In [20]:
from ingest_gao import build_chunks

chunks = build_chunks()

c = chunks[60]  # one chunk from the middle
print("METADATA:", c["metadata"], "\n")
print("TEXT (first 500 chars):\n", c["text"][:500])

Parsed PDF: 249 pages
Built 139 chunks (62 program profiles + 77 overview/appendix sections)
METADATA: {'program': 'FLRAA', 'page_start': 117, 'page_end': 118, 'source': 'GAO-26-108457 Weapon Systems Annual Assessment'} 

TEXT (first 500 chars):
 Army Program Type: MDAP Common Name: FLRAA 
 
Page 105 U.S. Government Accountability Office GAO-26-108457  Weapon Systems Annual Assessment 
 
Future Long Range Assault Aircraft (FLRAA)  
FLRAA is a top modernization priority for the Army. It is intended to be a 
medium-sized assault and utility aircraft and deliver speed, range, agility, 
endurance, and sustainability improvements as compared with current 
Black Hawk helicopters. The Army also expects the program to provide 
combatant commande


## 4. The vector store — semantic search in action

Each chunk was converted to an **embedding** (a vector of ~384 numbers capturing
its meaning) by a small free model running on your laptop, then stored in Chroma
(`data/chroma/`). Querying embeds your question the same way and returns the
*nearest* chunks — nearest in meaning, not just matching keywords.

Watch: the query below doesn't need the exact words used in the report.

In [21]:
from vector_store import VectorStore

store = VectorStore()
print(f"Chunks in the store: {store.count()}\n")

hits = store.query("fighter jet software delays", top_k=3)
for h in hits:
    m = h["metadata"]
    print(f"distance={h['distance']:.3f} | pages {m['page_start']}-{m['page_end']} | {m['program'][:60]}")

Chunks in the store: 139

distance=0.584 | pages 29-30 | GAO Weapon Systems Assessment (section)
distance=0.521 | pages 37-38 | GAO Weapon Systems Assessment (section)
distance=0.666 | pages 146-146 | DDG 1000


## 4b. The reranker — a second, sharper pass

`query()` now works in two stages: fast embeddings fetch a shortlist of 20,
then FlashRank (a cross-encoder) re-reads the question against each and
reorders them, keeping the best 4. The cell below shows the *before* order
(embeddings only) next to the *after* order (reranked) so you can see it work.

In [22]:
from vector_store import VectorStore

store = VectorStore()
q = "software development delays on fighter aircraft"

# STAGE 1 — peek at the raw embedding order by asking Chroma directly.
# (Reaching into store.collection here is just for illustration.)
raw = store.collection.query(query_texts=[q], n_results=8)
print("STAGE 1 — embedding order (what you'd get WITHOUT a reranker):")
for meta in raw["metadatas"][0][:5]:
    print(f"   pp.{meta['page_start']}-{meta['page_end']}  {meta['program'][:45]}")

# STAGE 2 — the normal query() runs FlashRank and returns the best top_k.
print("\nSTAGE 2 — after FlashRank reranking (what query() returns now):")
for h in store.query(q):
    m = h["metadata"]
    print(f"   score={h['rerank_score']:.2f}  pp.{m['page_start']}-{m['page_end']}  {m['program'][:45]}")

STAGE 1 — embedding order (what you'd get WITHOUT a reranker):
   pp.37-38  GAO Weapon Systems Assessment (section)
   pp.111-112  VC-25B
   pp.5-6  GAO Weapon Systems Assessment (section)
   pp.27-28  GAO Weapon Systems Assessment (section)
   pp.29-30  GAO Weapon Systems Assessment (section)

STAGE 2 — after FlashRank reranking (what query() returns now):
   score=0.60  pp.37-38  GAO Weapon Systems Assessment (section)
   score=0.60  pp.107-107  MH-139A
   score=0.45  pp.29-30  GAO Weapon Systems Assessment (section)
   score=0.32  pp.146-146  DDG 1000


## 5. A tool from the registry

`tools.py` holds the registry: each entry = a *definition* (what Claude reads to
decide when to use it) + a *function* (what we execute). `search_gao_reports`
wraps the vector store and formats results with `[Source: ...]` headers —
the raw material for citations.

In [23]:
from tools import get_tool_definitions, run_tool

print("Tools the agent can choose from:")
for d in get_tool_definitions():
    print(" -", d["function"]["name"])

print("\nRunning the search tool directly:\n")
result = run_tool("search_gao_reports", {"query": "aircraft carrier cost growth"})
print(result["text"][:900])
print("\nStructured citations:", result["citations"])

Tools the agent can choose from:
 - search_gao_reports
 - search_weapons_book

Running the search tool directly:

[Result 1 | section: CVN 78 | pages 145-145 | GAO-26-108457 Weapon Systems Annual Assessment]
Navy Program Type: MDAP Common Name: CVN 78 
 
 
 
Page 133 U.S. Government Accountability Office GAO-26-108457  Weapon Systems Annual Assessment 
   
Source: U.S. Navy.  |  GAO-26-108457 
CVN 78 Gerald R. Ford Nuclear Aircraft Carrier (CVN 78)  
The Navy developed the CVN 78 (or Ford class) nuclear-powered aircraft 
carrier to create operational efficiencies and increase the rate of sustained 
flight operations compared with legacy aircraft carriers. The Ford class 
introduced new propulsion, aircraft launch and recovery, and survivability 
capabilities to the carrier fleet. CVN 78 is the successor to the Nimitz class 
aircraft carriers. The Navy also expects the new technologies to enable 
Ford class carriers to operate with smaller crews than Nimitz class ships. 
 
 
Program Per

In [24]:
from tools import get_tool_definitions, run_tool

print("Tools the agent can choose from:")
for d in get_tool_definitions():
    print(" -", d["function"]["name"])

print("\nRunning the search tool directly:\n")
result = run_tool("search_gao_reports", {"query": "aircraft carrier cost growth"})
print(result["text"][:900])
print("\nStructured citations:", result["citations"])

Tools the agent can choose from:
 - search_gao_reports
 - search_weapons_book

Running the search tool directly:

[Result 1 | section: CVN 78 | pages 145-145 | GAO-26-108457 Weapon Systems Annual Assessment]
Navy Program Type: MDAP Common Name: CVN 78 
 
 
 
Page 133 U.S. Government Accountability Office GAO-26-108457  Weapon Systems Annual Assessment 
   
Source: U.S. Navy.  |  GAO-26-108457 
CVN 78 Gerald R. Ford Nuclear Aircraft Carrier (CVN 78)  
The Navy developed the CVN 78 (or Ford class) nuclear-powered aircraft 
carrier to create operational efficiencies and increase the rate of sustained 
flight operations compared with legacy aircraft carriers. The Ford class 
introduced new propulsion, aircraft launch and recovery, and survivability 
capabilities to the carrier fleet. CVN 78 is the successor to the Nimitz class 
aircraft carriers. The Navy also expects the new technologies to enable 
Ford class carriers to operate with smaller crews than Nimitz class ships. 
 
 
Program Per

## 6. The agent — OpenAI decides, we execute

Now the full loop (needs `OPENAI_API_KEY` in your `.env`):

1. OpenAI gets the question **plus the tool menu** → replies "call `search_gao_reports`"
2. We run it and hand back the chunks
3. OpenAI writes the answer **from the chunks**, with citations

Two calls to the API, one retrieval in between — that's the whole trick.

In [25]:
from agent import ask
import json

out = {}
for fragment in ask("What does GAO say about F-35 sustainment or modernization?", out=out):
    print(fragment, end="")
print(f"\n\n--- {len(out['tool_calls'])} tool call(s) made ---")
for tc in out["tool_calls"]:
    print("query used:", tc["input"])

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


The GAO assessment indicates that the F-35 program continues to face challenges in sustainment and modernization. Key points include:

1. **Sustainment and Cost Growth**: The program has encountered significant cost growth, which has raised concerns regarding its future sustainability (F-35 Joint Strike Fighter: More Actions Needed to Explain Cost Growth and Support Engine Modernization Decision, GAO-24-106047, pp. 12-13).

2. **Production Issues**: It is noted that the F-35, part of a broader strategy to deliver advanced capabilities to the Department of Defense, continues to encounter production issues that hamper straightforward modernization efforts (Related GAO Products, GAO-26-108457, pp. 244-245).

3. **Engine Modernization**: There are ongoing discussions on engine modernization which are critical to enhancing the operational capabilities of the F-35, with the need for clearer strategies to support such initiatives highlighted by the GAO (F-35 Joint Strike Fighter: More Actions

## 6b. Tracing — watching the agent think

Every call to `ask()` now logs a full trace to LangSmith: the prompt sent,
which tool was called, what chunks came back, and the final answer — with
timing and token cost on each step. This is what makes an agent debuggable
instead of a black box.

Run the cell above, then open https://smith.langchain.com, select the
**program-desk** project, and click the newest trace to see it laid out
step by step.

In [26]:
import os
from dotenv import load_dotenv
from langsmith import Client

load_dotenv()  # make sure .env is loaded even if this cell runs standalone

# --- Step 1: sanity-check tracing is actually configured ---
tracing_on = os.environ.get("LANGSMITH_TRACING", "").lower() == "true"
has_key = bool(os.environ.get("LANGSMITH_API_KEY"))
project_name = os.environ.get("LANGSMITH_PROJECT", "default")

print(f"LANGSMITH_TRACING   = {tracing_on}")
print(f"LANGSMITH_API_KEY set = {has_key}")
print(f"LANGSMITH_PROJECT   = {project_name!r}")

if not (tracing_on and has_key):
    print("\nTracing looks OFF or misconfigured.")
    print("   Check .env has LANGSMITH_TRACING=true and LANGSMITH_API_KEY=..., "
          "restart the kernel, run ask(...) once, then re-run this cell.")
else:
    client = Client()
    try:
        runs = list(client.list_runs(project_name=project_name, limit=5))
    except Exception as e:
        runs = []
        print(f"\nCouldn't reach LangSmith or find project {project_name!r}: {e}")

    if runs:
        runs.sort(key=lambda r: r.start_time, reverse=True)   # newest first
        latest = runs[0]
        url = client.get_run_url(run=latest)
        print(f"\nDirect link to your latest trace in {project_name!r}:")
        print(url)
    else:
        print(f"\nNo runs found yet in project {project_name!r}.")

LANGSMITH_TRACING   = True
LANGSMITH_API_KEY set = True
LANGSMITH_PROJECT   = 'US_Defense_Programs'


C:\Users\sol_v\AppData\Local\Temp\ipykernel_3992\4001773414.py:23: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  runs = list(client.list_runs(project_name=project_name, limit=5))



Direct link to your latest trace in 'US_Defense_Programs':
https://smith.langchain.com/o/a6cc1dad-5de2-439b-b86a-cf7b11340870/projects/p/66587ba0-f47a-45cb-898d-3382e0d7a635/r/01a039c2-f494-7070-a63e-678756d85085?poll=true


C:\Users\sol_v\AppData\Local\Temp\ipykernel_3992\4001773414.py:31: DeprecationWarning: get_run_url() is deprecated and will be removed after Jan 31, 2027. Use client.runs.get_url() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-get-url for the migration guide.
  url = client.get_run_url(run=latest)


## 7. The honesty test — refusing what the corpus can't answer

A core requirement: if the answer isn't in the corpus, the agent must say so
instead of guessing. Try something the GAO report doesn't cover:

In [27]:
out = {}
for fragment in ask("Who won the 2022 FIFA World Cup?", out=out):
    print(fragment, end="")
print(f"\n\n({len(out['tool_calls'])} tool call(s) made)")

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


I can't find this in the available corpus.

(0 tool call(s) made)



## 8. Where next week's features plug in

| Feature | Seam (look for the TODO comment) |
|---|---|
| Weapons Book / CRS ingestion | new `ingest_*.py` scripts, same shape as `ingest_gao.py` |
| USAspending live tool | one new entry in `tools.py` |
| Reranker | inside `VectorStore.query()` |
| Watchlist memory | `st.session_state` in `app.py` + agent context |
| LangSmith tracing | around the API call in `agent.py` |
| Chart understanding | new tool + image extraction at ingestion |
| Evaluation harness | new `eval.py` importing `ask()` directly |

Nothing above requires restructuring what you just walked through — that was the point.